In [ ]:
# AZURE SQL / RELATIONAL SOURCE INGESTION (SIMULATION)
# ----------------------------------------------------
# Goal:
# Simulate relational database ingestion into Fabric Bronze layer.
#
# What this notebook demonstrates:
# 1. Source-style relational tables (customers, orders, payments)
# 2. Bronze landing pattern
# 3. Real-world JDBC ingestion structure (as reference)
# 4. Validation of source and Bronze tables
#
# Important note:
# In this project, the relational source is simulated inside Fabric using tables.
# In a real environment, the same flow would read from Azure SQL / SQL Server
# using Spark JDBC.

###### <mark>**Create source-style DataFrames**</mark>

In [1]:
from pyspark.sql import Row

# -----------------------------------------
# STEP 1: CREATE SOURCE-STYLE RELATIONAL DATA
# -----------------------------------------
# These DataFrames simulate tables from an operational database source.

customers_data = [
    Row(customer_id=101, customer_name="Santhosh", email="santhosh@gmail.com", city="Bangalore"),
    Row(customer_id=102, customer_name="Rahul", email="rahul@gmail.com", city="Chennai"),
    Row(customer_id=103, customer_name="Anita", email="anita@gmail.com", city="Hyderabad"),
]

orders_data = [
    Row(order_id=9001, customer_id=101, product_id=3001, order_time="2026-04-13 15:30:00", order_amount=120.5, order_status="CREATED"),
    Row(order_id=9002, customer_id=102, product_id=3002, order_time="2026-04-13 15:31:00", order_amount=80.0, order_status="CREATED"),
    Row(order_id=9003, customer_id=103, product_id=3003, order_time="2026-04-13 15:32:00", order_amount=220.0, order_status="CREATED"),
]

payments_data = [
    Row(payment_id=5001, order_id=9001, payment_time="2026-04-13 15:33:00", payment_amount=120.5, payment_status="SUCCESS", payment_method="card"),
    Row(payment_id=5002, order_id=9002, payment_time="2026-04-13 15:34:00", payment_amount=80.0, payment_status="FAILED", payment_method="upi"),
    Row(payment_id=5003, order_id=9003, payment_time="2026-04-13 15:35:00", payment_amount=220.0, payment_status="SUCCESS", payment_method="wallet"),
]

# Create Spark DataFrames from source-style rows
df_customers_src = spark.createDataFrame(customers_data)
df_orders_src = spark.createDataFrame(orders_data)
df_payments_src = spark.createDataFrame(payments_data)

# Preview source DataFrames
display(df_customers_src)
display(df_orders_src)
display(df_payments_src)

StatementMeta(, 1928a340-bb96-497e-bd20-22573752a198, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6c4d0ec6-c056-4d7f-a4b0-f3fef1456fa1)

SynapseWidget(Synapse.DataFrame, 02e9a5bd-09b6-4abd-b901-2bc0adb7dd9f)

SynapseWidget(Synapse.DataFrame, b0b8fd19-705e-43a7-8920-c52ce0e27686)

###### <mark>**Write them as source tables**</mark>

In [4]:
# -----------------------------------------
# STEP 2: WRITE SOURCE TABLES
# -----------------------------------------
# These tables simulate relational source system tables.

df_customers_src.write.mode("overwrite").saveAsTable("sql_customers")
df_orders_src.write.mode("overwrite").saveAsTable("sql_orders")
df_payments_src.write.mode("overwrite").saveAsTable("sql_payments")

print("Simulated source tables created successfully.")

StatementMeta(, 1928a340-bb96-497e-bd20-22573752a198, 6, Finished, Available, Finished, False)

Simulated source tables created successfully.


###### <mark>**Validate in notebook**</mark>

In [5]:
# -----------------------------------------
# STEP 3: VALIDATE SOURCE TABLES
# -----------------------------------------

spark.sql("SELECT * FROM sql_customers").show(truncate=False)
spark.sql("SELECT * FROM sql_orders").show(truncate=False)
spark.sql("SELECT * FROM sql_payments").show(truncate=False)

StatementMeta(, 1928a340-bb96-497e-bd20-22573752a198, 7, Finished, Available, Finished, False)

+-----------+-------------+------------------+---------+
|customer_id|customer_name|email             |city     |
+-----------+-------------+------------------+---------+
|101        |Santhosh     |santhosh@gmail.com|Bangalore|
|103        |Anita        |anita@gmail.com   |Hyderabad|
|102        |Rahul        |rahul@gmail.com   |Chennai  |
+-----------+-------------+------------------+---------+

+--------+-----------+----------+-------------------+------------+------------+
|order_id|customer_id|product_id|order_time         |order_amount|order_status|
+--------+-----------+----------+-------------------+------------+------------+
|9003    |103        |3003      |2026-04-13 15:32:00|220.0       |CREATED     |
|9001    |101        |3001      |2026-04-13 15:30:00|120.5       |CREATED     |
|9002    |102        |3002      |2026-04-13 15:31:00|80.0        |CREATED     |
+--------+-----------+----------+-------------------+------------+------------+

+----------+--------+------------------

###### <mark>**Bronze ingestion**</mark>

In [6]:
# -----------------------------------------
# STEP 4: BRONZE INGESTION
# -----------------------------------------
# Read source-style tables and load them into Bronze tables.
# This simulates the result of a relational source ingestion flow.

bronze_customers_df = spark.read.table("sql_customers")
bronze_orders_df = spark.read.table("sql_orders")
bronze_payments_df = spark.read.table("sql_payments")

bronze_customers_df.write.mode("overwrite").saveAsTable("bronze_sql_customers")
bronze_orders_df.write.mode("overwrite").saveAsTable("bronze_sql_orders")
bronze_payments_df.write.mode("overwrite").saveAsTable("bronze_sql_payments")

print("Bronze SQL ingestion completed successfully.")

StatementMeta(, 1928a340-bb96-497e-bd20-22573752a198, 8, Finished, Available, Finished, False)

Bronze SQL ingestion completed successfully.


###### <mark>**Validate Bronze**</mark>

In [7]:
# -----------------------------------------
# STEP 5: VALIDATE BRONZE TABLES
# -----------------------------------------

spark.sql("SELECT * FROM bronze_sql_customers").show(truncate=False)
spark.sql("SELECT * FROM bronze_sql_orders").show(truncate=False)
spark.sql("SELECT * FROM bronze_sql_payments").show(truncate=False)

StatementMeta(, 1928a340-bb96-497e-bd20-22573752a198, 9, Finished, Available, Finished, False)

+-----------+-------------+------------------+---------+
|customer_id|customer_name|email             |city     |
+-----------+-------------+------------------+---------+
|101        |Santhosh     |santhosh@gmail.com|Bangalore|
|103        |Anita        |anita@gmail.com   |Hyderabad|
|102        |Rahul        |rahul@gmail.com   |Chennai  |
+-----------+-------------+------------------+---------+

+--------+-----------+----------+-------------------+------------+------------+
|order_id|customer_id|product_id|order_time         |order_amount|order_status|
+--------+-----------+----------+-------------------+------------+------------+
|9001    |101        |3001      |2026-04-13 15:30:00|120.5       |CREATED     |
|9003    |103        |3003      |2026-04-13 15:32:00|220.0       |CREATED     |
|9002    |102        |3002      |2026-04-13 15:31:00|80.0        |CREATED     |
+--------+-----------+----------+-------------------+------------+------------+

+----------+--------+------------------

In [8]:
# -----------------------------------------
# STEP 6: REAL-WORLD JDBC INGESTION TEMPLATE
# -----------------------------------------
# This is the pattern used in real projects to ingest from Azure SQL / SQL Server.
# It is included here as a learning and documentation reference.

jdbc_url = (
    "jdbc:sqlserver://<server-name>.database.windows.net:1433;"
    "database=<database-name>;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

jdbc_properties = {
    "user": "<username>",
    "password": "<password>",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

source_table = "dbo.orders"

# Real JDBC read example:
# df_orders_jdbc = (
#     spark.read
#     .format("jdbc")
#     .option("url", jdbc_url)
#     .option("dbtable", source_table)
#     .options(**jdbc_properties)
#     .load()
# )

print("JDBC ingestion template added for real-world Azure SQL reference.")

StatementMeta(, 1928a340-bb96-497e-bd20-22573752a198, 10, Finished, Available, Finished, False)

JDBC ingestion template added for real-world Azure SQL reference.


In [9]:
print("JDBC ingestion template added for real-world Azure SQL reference.")
print("In production, this will connect to Azure SQL / SQL Server using Spark JDBC.")

StatementMeta(, 1928a340-bb96-497e-bd20-22573752a198, 11, Finished, Available, Finished, False)

JDBC ingestion template added for real-world Azure SQL reference.
In production, this will connect to Azure SQL / SQL Server using Spark JDBC.
